In [1]:
#!pip install split-folders
#!pip install joblib
#!pip install skorch==1.4.0
#!pip install seaborn==0.13.2

In [2]:
import requests, zipfile, io, os, datetime, time, json, splitfolders, joblib
from urllib.parse import urlparse
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import is_classifier, is_regressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import CountVectorizer
import seaborn as sns
import matplotlib.pyplot as plt
from htb_ai_library import use_htb_style, HTB_GREEN, NODE_BLACK, HACKER_GREY
use_htb_style()
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.models as models
from torchvision import datasets, transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset
from utils import *
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
random_state=1337
torch.manual_seed(random_state)
HIDDEN_LAYER_SIZE = 1000

In [3]:
kdd_columns = ['duration','protocol_type','service','flag','src_bytes','dst_bytes','land','wrong_fragment','urgent','hot','num_failed_logins','logged_in','num_compromised','root_shell','su_attempted','num_root','num_file_creations','num_shells','num_access_files','num_outbound_cmds','is_host_login','is_guest_login','count','srv_count','serror_rate','srv_serror_rate','rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate','srv_diff_host_rate','dst_host_count','dst_host_srv_count','dst_host_same_srv_rate','dst_host_diff_srv_rate','dst_host_same_src_port_rate','dst_host_srv_diff_host_rate','dst_host_serror_rate','dst_host_srv_serror_rate','dst_host_rerror_rate','dst_host_srv_rerror_rate','attack','level']

attack_map = {1: ("dos_attacks", ["apache2","back","land","neptune","mailbomb","pod","processtable","smurf","teardrop","udpstorm","worm"]), 2: ("probe_attacks", ["ipsweep","mscan","nmap","portsweep","saint","satan"]), 3: ("privilege_attacks", ["buffer_overflow","loadmdoule","perl","ps","rootkit","sqlattack","xterm"]), 4: ("access_attacks", ["ftp_write","guess_passwd","http_tunnel","imap","multihop","named","phf","sendmail","snmpgetattack","snmpguess","spy","warezclient","warezmaster","xclock","xsnoop"])}

In [4]:
#--------------------------------------------------------------------------------------
# Orchestrating the Data Pipeline (Integration Into the ML Workflow):

def main(model_training_alg, data_path=None, custom_func=None, target_column=None, text_columns=None, header_names=None, dataset_type="csv", mean=None, std=None, model_type=None, plot=True, sep=",", n_epochs=None, n_channels=None, image_size=None,  X_train=None, X_val=None, X_test=None, y_train=None, y_val=None, y_test=None):

    """Main function for the model training pipeline."""
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {DEVICE}")
    ALG_CLASS_NAMES = {
        "random_forest": "RandomForestClassifier",
        "random_forest_regressor": "RandomForestRegressor",
        "linear": "LinearRegression",
        "naive_bayes": "MultinomialNB",
        "logistic_regression": "LogisticRegression"
    }
    dataset_type = dataset_type.lower()


    # Step 1: Load the data
    print("Loading dataset ...")
    if data_path and data_path.startswith("https://"):
      url_path = urlparse(data_path).path
      is_zip = url_path.lower().endswith(".zip") or "kaggle.com/api" in data_path
      data_path = fetch_dataset(data_path, zipped=is_zip)

      if dataset_type == "csv" and Path(data_path).is_dir():
          csv_files = list(Path(data_path).glob("*.csv"))
          if csv_files:
              data_path = csv_files[0]
          else:
              # no .csv extension in this archive — fall back to largest file
              files = [f for f in Path(data_path).iterdir() if f.is_file()]
              data_path = max(files, key=lambda f: f.stat().st_size)
          print(f"Using data file: {data_path.name}")

    if dataset_type == "image":
        train_loader, test_loader, n_classes, n_channels = load_data(data_path, dataset_type=dataset_type, mean=mean, std=std, image_size=image_size)
        model, training_output = train_model(model_training_alg="cnn", train_loader=train_loader, n_epochs=n_epochs, n_classes=n_classes, n_channels=n_channels, device=DEVICE)
        metrics = evaluate_nn(model, test_loader)
        if plot:
            # [Optional] Plot training details
            plot_training_accuracy(training_output)
            plot_training_loss(training_output)
        save_model(model, model_dir="saved_models", model_type="neuralnet")
        return model, metrics

    #elif dataset_type == "prepared":


    else:
        if dataset_type == "prepared":
            preprocessor = None
        else:
            df = load_data(data_path, header_names=header_names, dataset_type=dataset_type, mean=mean, std=std, sep=sep)
            if custom_func:
                df = custom_func(df)
            if not target_column:
                raise ValueError("target_column must be specified for non-image datasets")


            # Step 2: Preprocess the data
            print(f"\nPreprocessing the dataset ({df.shape[0]} Samples)...")
            X_train, X_val, X_test, y_train, y_val, y_test, preprocessor = preprocess_data(df, y_col_name=target_column, text_cols=text_columns)

            # Print processing results
            print(f"\nPreprocessing complete:")
            print(f"  - Training: {X_train.shape}")
            print(f"  - Validation: {X_val.shape}")
            print(f"  - Testing: {X_test.shape}")


        # Step 3: Train model with custom parameters (or load latest saved instance)
        class_name = ALG_CLASS_NAMES.get(model_training_alg)
        existing = sorted(f for f in glob.glob(f"saved_models/{class_name}_*.joblib") if "preprocessor" not in f)
        if existing:
            model_name = Path(existing[-1]).stem  # latest timestamp
            print(f"\nFound saved model '{model_name}', loading instead of training...")
            model, preprocessor, metadata = load_model_with_metadata(model_dir="saved_models", model_name=model_name)
        else:
            print("\n--- Training Model ---")
            model = train_model(X_train=X_train, y_train=y_train, model_training_alg=model_training_alg, preprocessor=preprocessor)


        # Step 4: Evaluate on validation (or test) set
        metrics, predictions, prediction_probabilities = evaluate_model(model, X_val, y_val)

        print("-" * 50)
        print("\n--- Evaluation Set Metrics ---")
        for metric_name, metric_value in metrics.items():
            if isinstance(metric_value, (int, float, np.floating)):
                print(f"\n  - {metric_name}: {metric_value:.4f}")
            else:
                print(f"\n  - {metric_name}:\n{metric_value}")

        # Generate predictions with the trained model (optional)
        y_pred, _ = predict_with_model(model, X_test)
        print(f"\nGenerated {len(y_pred)} predictions")
        if is_classifier(model):
            print(f"\nClassification Report: {classification_report(y_test, y_pred)}")

        # Save the model with rich metadata
        print("\n--- Saving Model ---")
        metadata = {
            "model_type": model.__class__.__name__,
            "metrics": metrics,
            "training_samples": X_train.shape[0],
            "feature_count": X_train.shape[1]
        }
        save_model(model, preprocessor, metadata)
        return model, metrics

In [7]:
# Diamonds — price regression
main(
    model_training_alg="random_forest_regressor",
    data_path="https://raw.githubusercontent.com/mwaskom/seaborn-data/master/diamonds.csv",
    dataset_type="csv",
    sep=",",
    target_column="price"
)

In [ ]:
# SMS Spam Collection — Spam Classification
main(
    model_training_alg="naive_bayes",
    data_path="https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip",
    target_column="label",
    text_columns=["message"],
    header_names=["label", "message"],
    dataset_type="csv",
    sep="\t"
)

In [ ]:
# KDD — Network Anomaly Detection
main(
    model_training_alg="random_forest",
    data_path="https://academy.hackthebox.com/storage/modules/292/KDD_dataset.zip",
    dataset_type="csv",
    sep=",",
    custom_func=prepare_kdd,
    target_column="attack_map",
    header_names=kdd_columns
)

In [ ]:
# malimg dataset - Malware classification

#import shutil
#shutil.rmtree("./newdata", ignore_errors=True)

main(
    model_training_alg="cnn",
    data_path="https://www.kaggle.com/api/v1/datasets/download/ikrambenabd/malimg-original",
    dataset_type="image",
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225],
    model_type="neuralnet",
    plot=True,
    n_epochs=10,
    image_size=75
)

In [ ]:
# Generate synthetic data for AI Data Attacks
n_samples = 1000
centers = [(0, 5), (5, 0)]  # Define centers for two distinct blobs
X, y = make_blobs(
    n_samples=n_samples,
    centers=centers,
    n_features=2,
    cluster_std=1.25,
    random_state=SEED,
)
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=SEED
)

print(f"Generated {n_samples} samples.")
print(f"Training set size: {X_train.shape[0]} samples.")
print(f"Testing set size: {X_test.shape[0]} samples.")
print(f"Number of features: {X_train.shape[1]}")
print(f"Classes: {np.unique(y)}")

model, metrics = main(model_training_alg="logistic_regression", X_train=X_train, X_val=X_test, X_test=X_test, y_train=y_train, y_val= y_test, y_test= y_test, dataset_type="prepared")